In [3]:
import argparse
import os
import json
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoImageProcessor, AutoModel
from tqdm import tqdm
import logging
from typing import Dict

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


In [4]:
import argparse
import os
import json
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoImageProcessor, AutoModel
from tqdm import tqdm
import logging
from typing import Dict
import torch.nn.functional as F # Needed for padding

# --- CKA Helper Functions (Adapted from NumPy version) ---

def gram_linear(x):
    """Compute Gram (kernel) matrix for a linear kernel.

    Args:
      x: A num_examples x num_features tensor of features.

    Returns:
      A num_examples x num_examples Gram matrix of examples.
    """
    return torch.matmul(x, x.T)

def center_gram(gram, unbiased=False):
    """Center a symmetric Gram matrix.

    Args:
      gram: A num_examples x num_examples symmetric tensor.
      unbiased: Whether to adjust the Gram matrix to compute an unbiased
        estimate of HSIC.

    Returns:
      A symmetric tensor with centered columns and rows.
    """
    if not torch.allclose(gram, gram.T):
        # Note: Floating point inaccuracies can sometimes lead to slight asymmetry.
        # Consider adding a tolerance check if needed: torch.allclose(gram, gram.T, atol=1e-6)
        # For now, we raise an error for clear non-symmetry.
        raise ValueError('Input must be a symmetric matrix.')
    gram = gram.clone()
    n = gram.shape[0]

    if unbiased:
        # This formulation of the U-statistic seems more numerically stable.
        # Equivalent to: H*gram*H where H = I - 1/n * 11^T - diag(1/n ... 1/n)
        # But uses adjusted means for stability.
        gram.fill_diagonal_(0)
        means = torch.sum(gram, dim=0) / (n - 2)
        means -= torch.sum(means) / (2 * (n - 1))
        gram -= means[:, None]
        gram -= means[None, :]
        gram.fill_diagonal_(0)
    else:
        # Equivalent to: H*gram*H where H = I - 1/n * 11^T
        means = torch.mean(gram, dim=0)
        means -= torch.mean(means) / 2
        gram -= means[:, None]
        gram -= means[None, :]

    return gram

def cka(gram_x, gram_y, debiased=False):
    """Compute CKA.

    Args:
      gram_x: A num_examples x num_examples Gram matrix.
      gram_y: A num_examples x num_examples Gram matrix.
      debiased: Use unbiased estimator of HSIC. CKA may still be biased.

    Returns:
      The value of CKA between X and Y.
    """
    gram_x = center_gram(gram_x, unbiased=debiased)
    gram_y = center_gram(gram_y, unbiased=debiased)

    # HSIC = trace(centered_gram_x @ centered_gram_y)
    # scaled_hsic = ||centered_gram_x @ centered_gram_y||_F^2 ??? No, dot product is correct.
    # The HCSIC computation simplifies for CKA calculation.
    scaled_hsic = torch.dot(gram_x.reshape(-1), gram_y.reshape(-1))

    normalization_x = torch.linalg.norm(gram_x, ord='fro')
    normalization_y = torch.linalg.norm(gram_y, ord='fro')
    
    # Handle potential division by zero
    if normalization_x == 0 or normalization_y == 0:
        return torch.tensor(0.0, device=gram_x.device) 
    
    return scaled_hsic / (normalization_x * normalization_y)

# --- End CKA Helper Functions ---

# --- CKA Computation Helper ---
def compute_and_print_cka(vit_embeddings_device, llm_hidden_states_device, batch_idx):
    """Computes and prints CKA between ViT embeddings and each LLM layer's hidden states.

    Args:
        vit_embeddings_device: Tensor of ViT embeddings on the compute device.
        llm_hidden_states_device: List of Tensors of LLM hidden states on the compute device.
        batch_idx: The current batch index (for printing).
    """
    if vit_embeddings_device is None or llm_hidden_states_device is None:
        logging.warning(f"Skipping CKA for batch {batch_idx} due to missing ViT or LLM embeddings.")
        return

    print(f"--- Batch {batch_idx} CKA Results (ViT vs LLM Layers) ---")
    I = vit_embeddings_device # Shape [N, Ti, Di]
    N, Ti, Di = I.shape
    
    for layer_idx, T in enumerate(llm_hidden_states_device):
        # T shape is [N, Tt, Dt]
        _, Tt, Dt = T.shape
        
        # Ensure N matches (batch size consistency)
        if I.shape[0] != T.shape[0]:
            logging.warning(f"Batch size mismatch between ViT ({I.shape[0]}) and LLM layer {layer_idx} ({T.shape[0]}). Skipping CKA for this layer.")
            continue

        # Method 1: Flattened CKA
        try:
            X_flat = I.reshape(N, -1) # [N, Ti * Di]
            Y_flat = T.reshape(N, -1) # [N, Tt * Dt]
            
            gram_x_flat = gram_linear(X_flat)
            gram_y_flat = gram_linear(Y_flat)
            
            cka_flattened_val = cka(gram_x_flat, gram_y_flat, debiased=True)
            print(f"  Layer {layer_idx}: CKA (Flattened) = {cka_flattened_val:.4f}")
        except Exception as e:
            print(f"  Layer {layer_idx}: CKA (Flattened) = ERROR ({e})")

        # Method 2: Padded CKA
        try:
            T_max = max(Ti, Tt)
            
            # Pad I (images) if Ti < T_max
            if Ti < T_max:
                padding_i = (0, 0, 0, T_max - Ti) # Pad last dim (tokens), then features
                Ipad = F.pad(I, padding_i, "constant", 0)
            else:
                Ipad = I
            
            # Pad T (text) if Tt < T_max
            if Tt < T_max:
                padding_t = (0, 0, 0, T_max - Tt) # Pad last dim (tokens), then features
                Tpad = F.pad(T, padding_t, "constant", 0)
            else:
                Tpad = T
            
            X_pad = Ipad.reshape(N, -1) # [N, T_max * Di]
            Y_pad = Tpad.reshape(N, -1) # [N, T_max * Dt]
            
            gram_x_pad = gram_linear(X_pad)
            gram_y_pad = gram_linear(Y_pad)
            
            cka_padded_val = cka(gram_x_pad, gram_y_pad, debiased=True)
            print(f"           CKA (Padded)   = {cka_padded_val:.4f}")
        except Exception as e:
            print(f"           CKA (Padded)   = ERROR ({e})")
    print("-----------------------------------------------------")

# --- End CKA Computation Helper ---

# --- Embedding Extraction Helper ---
def extract_embeddings_for_batch(batch, vit_model, llm_model, args, torch_dtype, batch_idx):
    """Performs forward passes and extracts embeddings for a single batch.

    Args:
        batch: Dictionary containing batch data (pixel_values, input_ids, etc.).
        vit_model: The Vision Transformer model.
        llm_model: The Language Model.
        args: Command-line arguments/configuration.
        torch_dtype: The torch data type to use.
        batch_idx: Current batch index (for logging).

    Returns:
        A tuple (vit_embeddings_device, llm_hidden_states_device).
        Returns (None, None) if extraction fails for either modality.
    """
    # Move batch to device (redundant if already done, but safe)
    try:
        pixel_values = batch['pixel_values'].to(args.device, dtype=torch_dtype) if batch['pixel_values'] is not None else None
        input_ids = batch['input_ids'].to(args.device) if batch['input_ids'] is not None else None
        attention_mask = batch['attention_mask'].to(args.device) if batch['attention_mask'] is not None else None
    except Exception as e:
        logging.error(f"Error moving batch {batch_idx} data to device {args.device} within extraction: {e}")
        return None, None

    # --- ViT Forward Pass ---
    vit_embeddings_device = None
    if pixel_values is not None:
        try:
            vit_outputs = vit_model(pixel_values=pixel_values)
            vit_embeddings_device = vit_outputs.last_hidden_state # Keep on device
        except Exception as e:
            logging.error(f"Error during ViT forward pass for batch {batch_idx}: {e}")
            # If ViT fails, we might still want LLM, but let's return None for simplicity now
            # return None, None # Option 1: Fail whole batch extraction
            vit_embeddings_device = None # Option 2: Continue and return None for ViT
    else:
        logging.warning(f"No valid images found in batch {batch_idx}. Skipping ViT forward pass.")

    # --- LLM Forward Pass ---
    llm_hidden_states_device = None
    if input_ids is not None and attention_mask is not None:
        try:
            llm_outputs = llm_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True
            )
            llm_hidden_states_device = llm_outputs.hidden_states # Keep on device
        except Exception as e:
            logging.error(f"Error during LLM forward pass for batch {batch_idx}: {e}")
            # If LLM fails, return None for LLM
            llm_hidden_states_device = None
    else:
         logging.warning(f"No valid text input found in batch {batch_idx}. Skipping LLM forward pass.")

    return vit_embeddings_device, llm_hidden_states_device

# --- End Embedding Extraction Helper ---

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --- Dataset Definition ---
class ImageTextDataset(Dataset):
    def __init__(self, data_path: str, image_folder: str):
        logging.info(f"Loading dataset index from: {data_path}")
        try:
            self.list_data_dict = json.load(open(data_path, "r"))
            # Basic validation of the first item
            if self.list_data_dict and isinstance(self.list_data_dict[0], dict):
                item = self.list_data_dict[0]
                if 'image' not in item or 'conversations' not in item:
                     raise ValueError("Dataset items must contain 'image' and 'conversations' keys.")
            elif not self.list_data_dict:
                 raise ValueError("Dataset JSON is empty.")
            else:
                raise ValueError("Dataset JSON format is not a list of dictionaries.")
            logging.info(f"Loaded {len(self.list_data_dict)} items from dataset index.")
        except Exception as e:
            logging.error(f"Error loading or parsing dataset index: {e}")
            raise
        self.image_folder = image_folder

    def __len__(self):
        return len(self.list_data_dict)

    def __getitem__(self, i) -> Dict[str, any]:
        item = self.list_data_dict[i]
        image_file = item.get('image')
        conversations = item.get('conversations', [])

        if not image_file:
            logging.warning(f"Missing 'image' key in item {i}. Skipping image.")
            image_file = None # Or handle differently

        # Concatenate conversation values to form the text
        text = "".join([conv.get('value', '') for conv in conversations])
        if not text:
             logging.warning(f"Empty text generated for item {i}.")

        return {
            "id": item.get('id', f'item_{i}'), # Use provided ID or generate one
            "image_file": image_file,
            "text": text.strip()
        }

# --- Collate Function ---
def create_collate_fn(tokenizer, image_processor, image_folder, device, model_max_length=512):
    def collate_fn(batch):
        image_files = [item['image_file'] for item in batch if item['image_file'] is not None]
        texts = [item['text'] for item in batch]
        ids = [item['id'] for item in batch]

        pixel_values = None
        if image_files:
            images = []
            valid_image_indices = [] # Keep track of which items in the batch have valid images
            for i, img_file in enumerate(image_files):
                 try:
                    img_path = os.path.join(image_folder, img_file)
                    images.append(Image.open(img_path).convert('RGB'))
                    valid_image_indices.append(i) # Assuming image loading/processing works for this index
                 except Exception as e:
                    logging.warning(f"Could not load image {img_file}: {e}. Skipping image for this item.")
            
            if images:
                try:
                    processed_images = image_processor(images=images, return_tensors='pt')
                    pixel_values = processed_images['pixel_values']
                except Exception as e:
                    logging.error(f"Error processing images with image_processor: {e}")
                    # Decide how to handle batch if image processing fails (e.g., return None, skip batch)
                    pixel_values = None # Or handle error differently

        try:
            tokenized_text = tokenizer(
                texts,
                return_tensors="pt",
                padding="longest",
                truncation=True,
                max_length=model_max_length # Use tokenizer's max length or a specific value
            )
        except Exception as e:
            logging.error(f"Error tokenizing texts: {e}")
            # Handle tokenization error (e.g., return None for text fields)
            tokenized_text = {'input_ids': None, 'attention_mask': None}

        return {
            'ids': ids,
            'image_files': image_files, # List of image files corresponding to pixel_values
            'pixel_values': pixel_values, # Tensor or None
            'input_ids': tokenized_text['input_ids'], # Tensor or None
            'attention_mask': tokenized_text['attention_mask'] # Tensor or None
        }
    return collate_fn

# --- Argument Parsing ---
def parse_args():
    parser = argparse.ArgumentParser(description="Extract embeddings from Vision Transformer and Language Model.")
    parser.add_argument("--model_name_or_path", type=str, 
                        default="mtgv/MobileLLaMA-1.4B-Base", 
                        help="Path or name of the pretrained language model.")
    parser.add_argument("--vision_tower", type=str, 
                        default="google/siglip-so400m-patch14-384", 
                        help="Path or name of the pretrained vision tower (e.g., CLIP).")
    parser.add_argument("--data_path", type=str, 
                        default="./playground/data/LLaVA-Pretrain/blip_laion_cc_sbu_558k.json", 
                        help="Path to the dataset JSON file.")
    parser.add_argument("--image_folder", type=str, 
                        default="./playground/data/LLaVA-Pretrain/images", 
                        help="Path to the folder containing images.")
    parser.add_argument("--output_dir", type=str, 
                        default="./extracted_embeddings", 
                        help="Directory to save the extracted embeddings.")
    parser.add_argument("--batch_size", type=int, default=4, 
                        help="Batch size for processing.")
    parser.add_argument("--model_max_length", type=int, default=2048, 
                        help="Maximum sequence length for the LLM tokenizer.")
    parser.add_argument("--max_batches", type=int, default=None, 
                        help="Maximum number of batches to process (optional).")
    parser.add_argument("--device", type=str, default="cuda" if torch.cuda.is_available() else "cpu", 
                        help="Device to use (cuda or cpu).")
    parser.add_argument("--dtype", type=str, default="bfloat16", 
                        choices=["float16", "bfloat16", "float32"], 
                        help="Data type for models (e.g., float16, bfloat16, float32).")
    # Pass an empty list to parse_args to avoid conflicts with kernel arguments
    args = parser.parse_args([]) 
    # Ensure output_dir is provided - Removing this check as default is set
    # if not args.output_dir:
    #     parser.error("--output_dir is required.")
    return args

# --- Helper Functions ---
def load_models(args):
    logging.info("Loading models and tokenizer...")
    # Determine torch dtype
    torch_dtype = torch.float32
    if args.dtype == "float16":
        torch_dtype = torch.float16
    elif args.dtype == "bfloat16":
        torch_dtype = torch.bfloat16
    logging.info(f"Using torch dtype: {torch_dtype}")

    # Load LLM Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(args.model_name_or_path, use_fast=False, padding_side="right")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        logging.warning("Tokenizer does not have a pad token. Setting pad_token to eos_token.")

    # Load LLM Model
    llm_model = AutoModelForCausalLM.from_pretrained(
        args.model_name_or_path,
        torch_dtype=torch_dtype
    ).to(args.device).eval()

    # Load ViT Processor
    image_processor = AutoImageProcessor.from_pretrained(args.vision_tower)

    # Load ViT Model
    vit_model = AutoModel.from_pretrained(
        args.vision_tower,
        torch_dtype=torch_dtype
    ).to(args.device).eval()
    if hasattr(vit_model, 'vision_model'):
        vit_model = vit_model.vision_model

    logging.info("Models loaded.")
    return tokenizer, llm_model, image_processor, vit_model, torch_dtype

def load_data(args, tokenizer, image_processor):
    logging.info("Loading dataset...")
    dataset = ImageTextDataset(data_path=args.data_path, image_folder=args.image_folder)
    collate_fn = create_collate_fn(
        tokenizer,
        image_processor,
        args.image_folder,
        args.device,
        model_max_length=args.model_max_length
    )
    dataloader = DataLoader(
        dataset,
        batch_size=args.batch_size,
        collate_fn=collate_fn,
        num_workers=4 # Adjust as needed
    )
    logging.info("Dataset loaded.")
    return dataloader

def process_batches(args, dataloader, llm_model, vit_model, torch_dtype):
    logging.info("Starting processing loop...")
    batch_count = 0
    total_items_processed = 0
    # Prepare subdirectories for outputs
    vit_output_dir = os.path.join(args.output_dir, "vit_embeddings")
    llm_output_dir = os.path.join(args.output_dir, "llm_hidden_states")
    os.makedirs(vit_output_dir, exist_ok=True)
    os.makedirs(llm_output_dir, exist_ok=True)

    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(dataloader, desc="Processing Batches")):
            if args.max_batches is not None and batch_idx >= args.max_batches:
                logging.info(f"Reached max_batches limit ({args.max_batches}). Stopping.")
                break

            # Move batch to device
            try:
                # We still need batch_ids here
                batch_ids = batch['ids']
                # Moving tensors to device is now handled within extract_embeddings_for_batch
            except Exception as e:
                logging.error(f"Error getting batch IDs for batch {batch_idx}: {e}")
                continue # Skip this batch
            
            # --- Embedding Extraction --- 
            vit_embeddings, llm_hidden_states_device = extract_embeddings_for_batch(
                batch, vit_model, llm_model, args, torch_dtype, batch_idx
            )

            # --- CKA Computation --- 
            compute_and_print_cka(vit_embeddings, llm_hidden_states_device, batch_idx)

            # --- Process Outputs (Print for Testing) --- (Modified to show shapes)
            print(f"--- Batch {batch_idx} --- Artifact Shapes --- (Embeddings kept on {args.device}) ---")
            if vit_embeddings is not None:
                print(f"  ViT Embeddings Shape: {vit_embeddings.shape}")
                # print(f"  ViT Embeddings (Batch 0, Patch 0, First 5 values):\n{vit_embeddings[0, 0, :5]}...") # Keep commented unless debugging
            else:
                print("  ViT Embeddings: Not computed or error occurred.")

            if llm_hidden_states_device is not None:
                print(f"  LLM Hidden States: {len(llm_hidden_states_device)} layers (including embeddings)")
                print(f"    Layer 0 (Embeddings) Shape: {llm_hidden_states_device[0].shape}")
                print(f"    Layer {len(llm_hidden_states_device)-1} (Final) Shape: {llm_hidden_states_device[-1].shape}")
                # print(f"    LLM Layer 0 (Batch 0, Token 0, First 5 values):\n{llm_hidden_states_device[0][0, 0, :5]}...") # Keep commented unless debugging
                # print(f"    LLM Layer {len(llm_hidden_states_device)-1} (Batch 0, Token 0, First 5 values):\n{llm_hidden_states_device[-1][0, 0, :5]}...") # Keep commented unless debugging
            else:
                print("  LLM Hidden States: Not computed or error occurred.")
            print("---------------------------------------")

            batch_count += 1
            total_items_processed += len(batch_ids)

    logging.info(f"Processing finished. Processed {batch_count} batches ({total_items_processed} items). Outputs saved to {args.output_dir}")


In [5]:
args = parse_args()
# <<<--- Add this line for testing on a small sample --->>>
args.max_batches = 1 # Process only the first batch for testing
logging.warning(f"*** TEST MODE: Limiting processing to {args.max_batches} batch(es) ***")
# <<<--------------------------------------------------->>>

logging.info(f"Starting embedding extraction with args: {args}")

# Create output directory
os.makedirs(args.output_dir, exist_ok=True)



2025-05-08 03:03:35,557 - WARNING - *** TEST MODE: Limiting processing to 1 batch(es) ***
2025-05-08 03:03:35,558 - INFO - Starting embedding extraction with args: Namespace(model_name_or_path='mtgv/MobileLLaMA-1.4B-Base', vision_tower='google/siglip-so400m-patch14-384', data_path='./playground/data/LLaVA-Pretrain/blip_laion_cc_sbu_558k.json', image_folder='./playground/data/LLaVA-Pretrain/images', output_dir='./extracted_embeddings', batch_size=4, model_max_length=2048, max_batches=1, device='cuda', dtype='bfloat16')


In [6]:
tokenizer, llm_model, image_processor, vit_model, torch_dtype = load_models(args)


2025-05-08 03:03:35,567 - INFO - Loading models and tokenizer...
2025-05-08 03:03:35,569 - INFO - Using torch dtype: torch.bfloat16
/opt/conda/envs/llava/lib/python3.10/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
2025-05-08 03:03:35,866 - WARNING - Tokenizer does not have a pad token. Setting pad_token to eos_token

g++ (Ubuntu 11.4.0-1ubuntu1~22.04) 11.4.0
Copyright (C) 2021 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

[2025-05-08 03:03:38,229] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)


2025-05-08 03:03:38,264 - INFO - gcc -pthread -B /opt/conda/envs/llava/compiler_compat -Wno-unused-result -Wsign-compare -DNDEBUG -fwrapv -O2 -Wall -fPIC -O2 -isystem /opt/conda/envs/llava/include -fPIC -O2 -isystem /opt/conda/envs/llava/include -fPIC -c /tmp/tmpy1fd2o6_/test.c -o /tmp/tmpy1fd2o6_/test.o
2025-05-08 03:03:38,298 - INFO - gcc -pthread -B /opt/conda/envs/llava/compiler_compat /tmp/tmpy1fd2o6_/test.o -laio -o /tmp/tmpy1fd2o6_/a.out
/opt/conda/envs/llava/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/opt/conda/envs/llava/lib/python3.10/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
2025-05-08 03:03:49,521 - INFO - Models loaded.


In [7]:
dataloader = load_data(args, tokenizer, image_processor)


2025-05-08 03:03:49,530 - INFO - Loading dataset...
2025-05-08 03:03:49,532 - INFO - Loading dataset index from: ./playground/data/LLaVA-Pretrain/blip_laion_cc_sbu_558k.json
2025-05-08 03:03:52,216 - INFO - Loaded 558128 items from dataset index.
2025-05-08 03:03:52,218 - INFO - Dataset loaded.


In [8]:
process_batches(args, dataloader, llm_model, vit_model, torch_dtype)

logging.info("Embedding extraction complete.")

2025-05-08 03:03:52,225 - INFO - Starting processing loop...
Processing Batches:   0%|          | 0/139532 [00:00<?, ?it/s]/opt/conda/envs/llava/lib/python3.10/site-packages/transformers/models/llama/modeling_llama.py:728: UserWarning: 1Torch was not compiled with memory efficient attention. (Triggered internally at /var/lib/jenkins/pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:505.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


--- Batch 0 CKA Results (ViT vs LLM Layers) ---
  Layer 0: CKA (Flattened) = -0.9805
           CKA (Padded)   = -0.9805
  Layer 1: CKA (Flattened) = -0.8711
           CKA (Padded)   = -0.8711
  Layer 2: CKA (Flattened) = -0.9336
           CKA (Padded)   = -0.9336
  Layer 3: CKA (Flattened) = -0.9609
           CKA (Padded)   = -0.9609
  Layer 4: CKA (Flattened) = -0.5352
           CKA (Padded)   = -0.5352
  Layer 5: CKA (Flattened) = -0.1670
           CKA (Padded)   = -0.1670
  Layer 6: CKA (Flattened) = 0.0908
           CKA (Padded)   = 0.0908
  Layer 7: CKA (Flattened) = 0.0908
           CKA (Padded)   = 0.0908
  Layer 8: CKA (Flattened) = -0.1670
           CKA (Padded)   = -0.1670
  Layer 9: CKA (Flattened) = -0.1670
           CKA (Padded)   = -0.1670
  Layer 10: CKA (Flattened) = -0.1670
           CKA (Padded)   = -0.1670
  Layer 11: CKA (Flattened) = -0.1670
           CKA (Padded)   = -0.1670
  Layer 12: CKA (Flattened) = 0.0908
           CKA (Padded)   = 0.0908
  Laye

Processing Batches:   0%|          | 1/139532 [00:06<259:49:14,  6.70s/it]
2025-05-08 03:03:58,935 - INFO - Processing finished. Processed 1 batches (4 items). Outputs saved to ./extracted_embeddings


  Layer 24: CKA (Flattened) = -0.9805
           CKA (Padded)   = -0.9805
-----------------------------------------------------
--- Batch 0 --- Artifact Shapes --- (Embeddings kept on cuda) ---
  ViT Embeddings Shape: torch.Size([4, 729, 1152])
  LLM Hidden States: 25 layers (including embeddings)
    Layer 0 (Embeddings) Shape: torch.Size([4, 46, 2048])
    Layer 24 (Final) Shape: torch.Size([4, 46, 2048])
---------------------------------------


2025-05-08 03:03:58,938 - INFO - Embedding extraction complete.


In [9]:
# def main():
args = parse_args()
logging.info(f"Starting embedding extraction with args: {args}")

# <<<--- Add this line for testing on a small sample --->>>
args.max_batches = 1 # Process only the first batch for testing
logging.warning(f"*** TEST MODE: Limiting processing to {args.max_batches} batch(es) ***")
# <<<--------------------------------------------------->>>

# Create output directory
os.makedirs(args.output_dir, exist_ok=True)

# --- Load Models and Tokenizer ---
tokenizer, llm_model, image_processor, vit_model, torch_dtype = load_models(args)

# --- Load Data ---
dataloader = load_data(args, tokenizer, image_processor)

# --- Process Batches --- 
process_batches(args, dataloader, llm_model, vit_model, torch_dtype)

logging.info("Embedding extraction complete.")

# if __name__ == "__main__":
#     main() 

2025-05-08 03:03:58,952 - INFO - Starting embedding extraction with args: Namespace(model_name_or_path='mtgv/MobileLLaMA-1.4B-Base', vision_tower='google/siglip-so400m-patch14-384', data_path='./playground/data/LLaVA-Pretrain/blip_laion_cc_sbu_558k.json', image_folder='./playground/data/LLaVA-Pretrain/images', output_dir='./extracted_embeddings', batch_size=4, model_max_length=2048, max_batches=None, device='cuda', dtype='bfloat16')
2025-05-08 03:03:58,953 - WARNING - *** TEST MODE: Limiting processing to 1 batch(es) ***
2025-05-08 03:03:58,955 - INFO - Loading models and tokenizer...
2025-05-08 03:03:58,955 - INFO - Using torch dtype: torch.bfloat16
2025-05-08 03:03:59,098 - WARNING - Tokenizer does not have a pad token. Setting pad_token to eos_token.
2025-05-08 03:04:00,473 - INFO - Models loaded.
2025-05-08 03:04:00,476 - INFO - Loading dataset...
2025-05-08 03:04:00,476 - INFO - Loading dataset index from: ./playground/data/LLaVA-Pretrain/blip_laion_cc_sbu_558k.json
2025-05-08 03:

--- Batch 0 CKA Results (ViT vs LLM Layers) ---
  Layer 0: CKA (Flattened) = -0.9805
           CKA (Padded)   = -0.9805
  Layer 1: CKA (Flattened) = -0.8711
           CKA (Padded)   = -0.8711
  Layer 2: CKA (Flattened) = -0.9336
           CKA (Padded)   = -0.9336
  Layer 3: CKA (Flattened) = -0.9609
           CKA (Padded)   = -0.9609
  Layer 4: CKA (Flattened) = -0.5352
           CKA (Padded)   = -0.5352
  Layer 5: CKA (Flattened) = -0.1670
           CKA (Padded)   = -0.1670
  Layer 6: CKA (Flattened) = 0.0908
           CKA (Padded)   = 0.0908
  Layer 7: CKA (Flattened) = 0.0908
           CKA (Padded)   = 0.0908
  Layer 8: CKA (Flattened) = -0.1670
           CKA (Padded)   = -0.1670
  Layer 9: CKA (Flattened) = -0.1670
           CKA (Padded)   = -0.1670
  Layer 10: CKA (Flattened) = -0.1670
           CKA (Padded)   = -0.1670
  Layer 11: CKA (Flattened) = -0.1670
           CKA (Padded)   = -0.1670
  Layer 12: CKA (Flattened) = 0.0908
           CKA (Padded)   = 0.0908
  Laye

Processing Batches:   0%|          | 1/139532 [00:04<167:45:47,  4.33s/it]2025-05-08 03:04:08,185 - INFO - Reached max_batches limit (1). Stopping.


           CKA (Padded)   = -0.9805
-----------------------------------------------------
--- Batch 0 --- Artifact Shapes --- (Embeddings kept on cuda) ---
  ViT Embeddings Shape: torch.Size([4, 729, 1152])
  LLM Hidden States: 25 layers (including embeddings)
    Layer 0 (Embeddings) Shape: torch.Size([4, 46, 2048])
    Layer 24 (Final) Shape: torch.Size([4, 46, 2048])
---------------------------------------


Processing Batches:   0%|          | 1/139532 [00:04<178:59:08,  4.62s/it]
2025-05-08 03:04:08,473 - INFO - Processing finished. Processed 1 batches (4 items). Outputs saved to ./extracted_embeddings
2025-05-08 03:04:08,477 - INFO - Embedding extraction complete.
